# Aralin 11 - Model Context Protocol (MCP)

Ang **Model Context Protocol (MCP)** ay isang bukas na pamantayan na nagpapahintulot sa mga ahente na matuklasan at gamitin nang dinamiko ang mga kasangkapan, mapagkukunan, at pinagmumulan ng datos habang tumatakbo. Sa halip na i-hardcode ang mga kasangkapan sa isang ahente, pinapayagan ng MCP ang mga ahente na kumonekta sa mga panlabas na server na nagbibigay ng mga kakayahan ayon sa pangangailangan.

Sa araling ito, matututuhan mo:
- Ano ang MCP at bakit ito mahalaga para sa mga sistema ng ahente
- Paano gumagana ang arkitekturang kliyente-server ng MCP
- Paano bumuo ng mga ahente na gumagamit ng pagtuklas ng mga kasangkapan gaya ng MCP


## Pagsasaayos

**Mga Kinakailangan:**
- Proyekto ng Azure AI Foundry na may naka-deploy na modelo
- Patakbuhin ang `az login` para sa awtentikasyon


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ano ang Model Context Protocol (MCP)?

Nagbibigay ang MCP ng isang pamantayang paraan para matuklasan at makipag-ugnayan ng mga AI agent sa mga panlabas na tool at pinagkukunan ng datos:

- **MCP Server**: Naglalantad ng mga tool, mga mapagkukunan, at mga prompt sa pamamagitan ng isang pamantayang protocol
- **MCP Client**: Ang runtime ng ahente na kumokonekta sa mga server at natutuklasan ang mga magagamit na kakayahan
- **Dynamic Discovery**: Hindi kailangan ng mga ahente ng hardcoded na mga tool — natutuklasan nila kung ano ang magagamit habang tumatakbo

Makapangyarihan ito para sa pagbuo ng mga sistemang ahente na madaling mapapalawig, kung saan maaaring idagdag ang mga bagong kakayahan nang hindi binabago ang code ng ahente.


## Paano Gumagana ang MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Ang ahente (MCP client) ay kumokonekta sa isang MCP server
2. Tumutugon ang server ng isang listahan ng mga magagamit na tool at ng kanilang mga schema
3. Maaari nang tumawag ang ahente sa anumang natuklasang tool habang nag-iisip ito
4. Bumabalik ang mga resulta sa pamamagitan ng parehong protocol


## Pagsasagaya ng Pagtuklas ng Mga Tool ng MCP

Dahil ang isang totoong MCP server ay nangangailangan ng isang tumatakbong proseso ng server, ipapakita namin ang pattern gamit ang mga `@tool` na function na ginagaya kung ano ang ibibigay ng isang accommodation service na konektado sa MCP.

Sa produksyon, ang mga tool na ito ay matutuklasan nang dinamiko mula sa isang MCP server sa halip na ideklara nang lokal.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Pagbuo ng Ahente gamit ang mga Kasangkapan na Estilo MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP sa Produksyon

Sa isang kapaligiran ng produksyon, pinapagana ng MCP ang mga makapangyarihang pattern:

- **Dinamikong pagtuklas ng mga tool**: Kumokonekta ang mga ahente sa mga MCP server at natutuklasan ang mga tool habang tumatakbo
- **Nakahiwalay na arkitektura**: Maaaring mag-update ang mga provider ng tool nang hiwalay sa ahente
- **Pagbabahagi sa pagitan ng mga organisasyon**: Maaaring ilantad ng mga koponan ang mga kakayahan sa pamamagitan ng mga MCP server na maaaring gamitin ng anumang ahente
- **Suporta ng Microsoft Agent Framework**: May nakapaloob na suporta ang MAF para sa MCP client sa pamamagitan ng `mcp` integration

Upang gumamit ng totoong MCP server kasama ang MAF, magkokonekta ka sa pamamagitan ng `hosted_mcp_tool()` o ng MCP client integration.

**Alamin pa:**
- [Spesipikasyon ng MCP](https://modelcontextprotocol.io/)
- [Suporta ng MCP ng Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Buod

Sa leksyong ito, natutunan mo:
- **MCP** ay isang bukas na pamantayan para sa dinamikong pagtuklas ng mga kasangkapan sa pagitan ng mga ahente at mga tagapagbigay ng kasangkapan
- Ang **client-server architecture** ay nagpapahintulot sa mga ahente na matuklasan ang mga kakayahan sa runtime
- Pinapagana ng MCP ang mga **napapalawak at hiwalay na sistema ng ahente** kung saan maaaring idagdag ang mga tool nang hindi binabago ang code
- Nagbibigay ang Microsoft Agent Framework ng **nakapaloob na suporta sa MCP** para sa paggamit sa produksyon


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Paunawa:
Isinalin ang dokumentong ito gamit ang serbisyong pagsasalin na pinapagana ng AI na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagaman nagsusumikap kami na maging tumpak, pakitandaan na ang mga awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatumpak. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pinakapinagkakatiwalaang sanggunian. Para sa mga kritikal na impormasyon, inirerekomenda ang propesyonal na pagsasalin na ginawa ng tao. Hindi kami mananagot sa anumang mga hindi pagkakaunawaan o maling interpretasyon na maaaring magmula sa paggamit ng salin na ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
